# Setup: clone the project (Colab) and install requirements

In [ ]:
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    # Trained models are cached in this Drive folder between sessions
    DRIVE_BASE = '/content/drive/MyDrive/brain-glioma-classifier'
    REPO_DIR = '/content/brain-glioma-classifier'

    if os.path.exists(REPO_DIR):
        # already cloned — just pull latest changes
        !git -C {REPO_DIR} pull
    else:
        # fresh session — clone from scratch
        !git clone https://github.com/yotamcooper/brain-glioma-classifier.git {REPO_DIR}

    %cd {REPO_DIR}
    !pip install -r requirements.txt -q
else:
    # Running locally from the repo folder: models/ and figures/ stay on this machine
    DRIVE_BASE = None

### Getting the data

In [ ]:
import json
from src.preprocessing import prepare_stage1_data, prepare_stage2_data

# Both datasets are downloaded from their public sources on the first run,
# so nothing has to be uploaded to Drive beforehand.
#   Stage 2: UCI glioma grading table   -> data/TCGA_GBM_LGG_Mutations_all.csv
#   Stage 1: original Mendeley MRI scans -> train / val / test split
# The split uses the ORIGINAL images only (not the pre-augmented tumor folder)
# and drops byte-identical duplicates, so no scan sits in two splits.
csv_path = prepare_stage2_data()
data_dir = prepare_stage1_data()

with open(os.path.join(data_dir, 'split_info.json')) as f:
    split_info = json.load(f)
print(f"Glioma table : {csv_path}")
print(f"MRI split    : {split_info['counts']}")
print(f"Duplicates removed: {split_info['duplicates_removed']}")
print("Data ready ✅")

## Stage 1 cell

In [ ]:
from src.pipeline import run_stage1
from src.drive_cache import restore, store

# A saved model is reused only if the current version of the pipeline trained it.
# Anything older (e.g. trained on the augmented split) is ignored and Stage 1
# retrains — use a GPU runtime for that, it is slow on CPU.
cached = restore('stage1', DRIVE_BASE)

model, history = run_stage1(load_saved=cached)

if cached:
    print("✅ Stage 1 loaded from saved model")
else:
    store('stage1', DRIVE_BASE)
    print("✅ Stage 1 trained and saved")

## Stage 2 cell

In [ ]:
from src.pipeline import run_stage2
from src.drive_cache import restore, store

cached = restore('stage2', DRIVE_BASE)

stage2_metrics = run_stage2(load_saved=cached)

if cached:
    print("✅ Stage 2 loaded from saved model")
else:
    store('stage2', DRIVE_BASE)
    print("✅ Stage 2 trained and saved")

## Stage 1 Predictions

In [ ]:
# Testing Stage 1 predict
# needs a folder that has the images in; you can change the path to whatever
from src.predict import predict_tumor

result = predict_tumor("data/Brain Tumor MRI Dataset/test/Tumor", model=model)
print({k: v for k, v in result.items() if k != "scan_results"})

## Stage 2 Predictions

In [ ]:
# Testing Stage 2 predict
# the input is a dict so whatever patient details you give itll predict
from src.predict import predict_grading

result = predict_grading({
    "Age_at_diagnosis": "52 years 180 days",
    "Gender": "Male",
    "Race": "white",
    "IDH1": "MUTATED",
    "TP53": "NOT_MUTATED",
    "ATRX": "NOT_MUTATED",
    "PTEN": "NOT_MUTATED",
    "EGFR": "NOT_MUTATED",
    "CIC": "NOT_MUTATED",
    "MUC16": "NOT_MUTATED",
    "PIK3CA": "NOT_MUTATED",
    "NF1": "NOT_MUTATED",
    "PIK3R1": "NOT_MUTATED",
    "FUBP1": "NOT_MUTATED",
    "RB1": "NOT_MUTATED",
    "NOTCH1": "NOT_MUTATED",
    "BCOR": "NOT_MUTATED",
    "CSMD3": "NOT_MUTATED",
    "SMARCA4": "NOT_MUTATED",
    "GRIN2A": "NOT_MUTATED",
    "IDH2": "NOT_MUTATED",
    "FAT4": "NOT_MUTATED",
    "PDGFRA": "NOT_MUTATED",
})
print(result)